# Lanjutan evaluasi

In [31]:
EVALUATION_ONLY = True

# Eksperimen Instance Segmentation Indoor

**NRP 041 · Panoptic Indoor Segmentation v5**

Membandingkan YOLOv8n-seg, YOLOv8s-seg, dan Mask R-CNN ResNet-50 FPN pada ukuran 320, 640, dan 960. Setiap konfigurasi memiliki sel training sendiri. Run dimulai dari 640, kemudian 320 dan 960 dalam setiap bagian model.

| Model | Keluarga | Ukuran input | Epoch |
|---|---|---|---|
| YOLOv8n-seg | One-stage | 640, 320, 960 | 30 |
| YOLOv8s-seg | One-stage | 640, 320, 960 | 30 |
| Mask R-CNN ResNet-50 FPN | Two-stage | 640, 320, 960 | 30 |


In [32]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow", "ultralytics", "pycocotools"])

0

In [33]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Pilih GPU di pengaturan Accelerator Kaggle, lalu jalankan ulang."
print("GPU:", torch.cuda.get_device_name(0))
from ultralytics import YOLO
import torchvision
print("Torchvision:", torchvision.__version__)

PyTorch: 2.11.0+cu130
CUDA tersedia: True
GPU: Tesla T4
Torchvision: 0.26.0+cu130


### Unduh dataset resmi v5


In [34]:
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

WORK_DIR = Path("/kaggle/working")
RUNS_DIR = WORK_DIR / "visi_komputer_041"
rf = Roboflow(api_key="7f7iWApDwJRfabLV1SCX")
project = rf.workspace("panopticindoor").project("panoptic-indoor-segmentation")
dataset = project.version(5).download(
    "coco-segmentation", location=str(WORK_DIR / "panoptic_indoor_coco_v5")
)

loading Roboflow workspace...
loading Roboflow project...


### Periksa split dan anotasi mask


In [35]:
import json
from pathlib import Path

root = Path(dataset.location)

for split in ("train", "valid", "test"):
    path = root / split / "_annotations.coco.json"
    if not path.is_file():
        print(f"{split}: file anotasi belum ditemukan di {path}")
        continue

    data = json.loads(path.read_text())
    masks = sum(bool(a.get("segmentation")) for a in data["annotations"])
    print(f"{split}: {len(data['images'])} gambar, {masks} anotasi dengan mask")

train: 397 gambar, 4479 anotasi dengan mask
valid: 166 gambar, 2022 anotasi dengan mask
test: 359 gambar, 5344 anotasi dengan mask


### Siapkan format YOLO Segmentation

In [36]:
import json
import shutil
from collections import defaultdict
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import yaml
from ultralytics.data.converter import merge_multi_segment

root = Path(dataset.location)
converted = Path(mkdtemp(prefix="panoptic_indoor_yolo_", dir=str(WORK_DIR)))
split_counts = {"train": 397, "valid": 166, "test": 359}

data_by_split = {
    split: json.loads(
        (root / split / "_annotations.coco.json").read_text(encoding="utf-8")
    )
    for split in split_counts
}

categories = sorted(
    data_by_split["train"]["categories"], key=lambda category: category["id"]
)
names = [category["name"] for category in categories]
class_by_name = {name: index for index, name in enumerate(names)}

for split, data in data_by_split.items():
    if len(data["images"]) != split_counts[split]:
        raise ValueError(f"{split}: jumlah gambar berbeda dari split resmi v5.")

    class_ids = {
        category["id"]: class_by_name[category["name"]]
        for category in data["categories"]
    }

    annotations_by_image = defaultdict(list)
    for ann in data["annotations"]:
        if not ann.get("iscrowd"):
            annotations_by_image[ann["image_id"]].append(ann)

    mask_count = 0
    duplicate_count = 0

    for img in data["images"]:
        filename = Path(img["file_name"])
        image_path = converted / "images" / split / filename
        label_path = (converted / "labels" / split / filename).with_suffix(".txt")
        rows = []
        seen = set()

        for ann in annotations_by_image[img["id"]]:
            polygons = ann.get("segmentation")
            if not isinstance(polygons, list) or not polygons or any(
                not isinstance(polygon, list)
                or len(polygon) < 6
                or len(polygon) % 2
                for polygon in polygons
            ):
                raise ValueError(f"{split}/{filename}: mask ID {ann['id']} tidak valid.")

            class_id = class_ids[ann["category_id"]]
            key = (class_id, tuple(tuple(polygon) for polygon in polygons))
            if key in seen:
                duplicate_count += 1
                continue
            seen.add(key)

            if len(polygons) == 1:
                points = np.asarray(polygons[0], dtype=float).reshape(-1, 2)
            else:
                points = np.concatenate(merge_multi_segment(polygons), axis=0)

            points = points / [img["width"], img["height"]]
            if not np.isfinite(points).all() or (points < 0).any() or (points > 1).any():
                raise ValueError(
                    f"{split}/{filename}: koordinat mask ID {ann['id']} di luar gambar."
                )

            coordinates = " ".join(f"{value:.10g}" for value in points.ravel())
            rows.append(f"{class_id} {coordinates}")

        image_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(root / split / filename, image_path)
        label_path.write_text("\n".join(rows), encoding="utf-8")
        mask_count += len(rows)

    print(
        f"{split}: {len(data['images'])} gambar, {mask_count} mask, "
        f"{duplicate_count} duplikat persis dilewati"
    )

yaml_path = converted / "data.yaml"
yaml_path.write_text(
    yaml.safe_dump({
        "path": str(converted),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": names,
    }, sort_keys=False),
    encoding="utf-8",
)

print(f"Dataset YOLO siap: {yaml_path}")


train: 397 gambar, 4479 mask, 0 duplikat persis dilewati
valid: 166 gambar, 2021 mask, 1 duplikat persis dilewati
test: 359 gambar, 5341 mask, 3 duplikat persis dilewati
Dataset YOLO siap: /kaggle/working/panoptic_indoor_yolo_3o0v5xc2/data.yaml


## Fungsi training dan penyimpanan

YOLO memakai batch 4, seed 41, dan pemilihan checkpoint dari fitness validasi bawaan Ultralytics. Mask R-CNN memakai batch fisik 1 dengan akumulasi gradien 4, SGD learning rate 0.005, dan memilih checkpoint berdasarkan COCO mask AP50–95 pada validasi.

In [37]:
import csv
import gc
import importlib.metadata
import random
import time

import torch

assert torch.cuda.is_available(), "Aktifkan GPU Kaggle sebelum menjalankan training."
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Catat versi pustaka dari sesi yang benar-benar menjalankan eksperimen.
versions = {}
for package in ("torch", "torchvision", "ultralytics", "roboflow", "numpy"):
    versions[package] = importlib.metadata.version(package)
(WORK_DIR / "training_environment.json").write_text(json.dumps(versions, indent=2))


def completed_run(name, checkpoint):
    candidates = [RUNS_DIR / name]
    candidates.extend(Path("/kaggle/input").glob(f"**/{name}/training_complete.json"))
    for candidate in candidates:
        directory = candidate.parent if candidate.is_file() else candidate
        marker = directory / "training_complete.json"
        if marker.is_file() and (directory / checkpoint).is_file():
            info = json.loads(marker.read_text())
            if info.get("epochs") == 30:
                print("Hasil training lengkap ditemukan:", directory)
                return directory
    return None


def finish_run(directory, info):
    info["epochs"] = 30
    info["versions"] = versions
    (directory / "training_complete.json").write_text(json.dumps(info, indent=2))
    archive = shutil.make_archive(str(WORK_DIR / directory.name), "zip", root_dir=directory)
    print("ZIP selesai:", archive, flush=True)


In [38]:
def train_yolo_run(weights, size, name):
    previous = completed_run(name, "weights/best.pt")
    if previous is not None:
        return previous

    started = time.time()
    network = YOLO(weights)
    network.train(
        data=str(yaml_path),
        imgsz=size,
        epochs=30,
        patience=0,
        batch=4,
        device=0,
        seed=41,
        project=str(RUNS_DIR),
        name=name,
        workers=2,
    )
    directory = Path(network.trainer.save_dir)
    finish_run(directory, {
        "model": weights, "input_size": size, "batch_size": 4,
        "seed": 41, "seconds": time.time() - started,
        "best_selection": "Ultralytics validation fitness",
    })
    del network
    gc.collect()
    torch.cuda.empty_cache()
    return directory


In [39]:
import contextlib
import copy
import io
import math

from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm


class IndoorCocoDataset(Dataset):
    def __init__(self, split, class_names):
        self.split = split
        self.image_dir = root / split
        source = json.loads((self.image_dir / "_annotations.coco.json").read_text())
        self.images = source["images"]
        self.foreground_id = {name: i + 1 for i, name in enumerate(class_names)}
        self.category_to_label = {
            cat["id"]: self.foreground_id[cat["name"]]
            for cat in source["categories"]
        }
        self.label_to_category = {value: key for key, value in self.category_to_label.items()}
        self.annotations = defaultdict(list)
        filtered = []
        seen = set()
        for ann in source["annotations"]:
            key = (ann["image_id"], ann["category_id"], json.dumps(ann["segmentation"]))
            if key in seen:
                continue
            seen.add(key)
            filtered.append(ann)
            self.annotations[ann["image_id"]].append(ann)

        source["annotations"] = filtered
        self.coco = COCO()
        self.coco.dataset = copy.deepcopy(source)
        self.coco.dataset.setdefault("info", {})
        self.coco.createIndex()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        info = self.images[index]
        with Image.open(self.image_dir / info["file_name"]) as image:
            image = to_tensor(image.convert("RGB"))
        height, width = image.shape[-2:]
        if (height, width) != (info["height"], info["width"]):
            raise ValueError(f"Ukuran gambar berbeda dari COCO: {info['file_name']}")
        boxes, labels, masks = [], [], []
        for ann in self.annotations[info["id"]]:
            if ann.get("iscrowd"):
                continue
            mask = self.coco.annToMask(ann).astype(np.uint8)
            if not mask.any():
                raise ValueError(f"Mask kosong pada ID {ann['id']}")
            x, y, w, h = ann["bbox"]
            box = [max(0, x), max(0, y), min(width, x + w), min(height, y + h)]
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError(f"BBox tidak valid pada ID {ann['id']}")
            boxes.append(box)
            labels.append(self.category_to_label[ann["category_id"]])
            masks.append(mask)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "masks": torch.from_numpy(np.stack(masks)) if masks else torch.zeros(
                (0, height, width), dtype=torch.uint8
            ),
            "image_id": torch.tensor(info["id"]),
        }
        return image, target


def collate_detection(batch):
    return tuple(zip(*batch))


train_dataset = IndoorCocoDataset("train", names)
valid_dataset = IndoorCocoDataset("valid", names)


def build_maskrcnn(size):
    network = maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,
        min_size=size, max_size=size,
        box_score_thresh=0.001, box_detections_per_img=100,
    )
    box_features = network.roi_heads.box_predictor.cls_score.in_features
    network.roi_heads.box_predictor = FastRCNNPredictor(box_features, len(names) + 1)
    mask_features = network.roi_heads.mask_predictor.conv5_mask.in_channels
    network.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, len(names) + 1)
    return network.cuda()


@torch.inference_mode()
def validation_mask_ap(network):
    network.eval()
    predictions = []
    for image, target in tqdm(valid_dataset, desc="Validasi mask", total=len(valid_dataset)):
        output = network([image.cuda()])[0]
        image_id = int(target["image_id"])
        for label, score, mask in zip(output["labels"], output["scores"], output["masks"]):
            binary = (mask[0] >= 0.5).cpu().numpy().astype(np.uint8)
            if not binary.any():
                continue
            encoded = mask_utils.encode(np.asfortranarray(binary))
            encoded["counts"] = encoded["counts"].decode("ascii")
            predictions.append({
                "image_id": image_id,
                "category_id": valid_dataset.label_to_category[int(label)],
                "segmentation": encoded,
                "score": float(score),
            })
        del output

    if not predictions:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        predicted_coco = valid_dataset.coco.loadRes(predictions)
        evaluation = COCOeval(valid_dataset.coco, predicted_coco, "segm")
        evaluation.params.imgIds = [image["id"] for image in valid_dataset.images]
        evaluation.evaluate()
        evaluation.accumulate()
        evaluation.summarize()
    return float(evaluation.stats[0]), float(evaluation.stats[1])


creating index...
index created!
creating index...
index created!


In [40]:
def train_maskrcnn_run(size):
    name = f"maskrcnn_r50_fpn_{size}_resmi"
    previous = completed_run(name, "best.pth")
    if previous is not None:
        return previous

    random.seed(41)
    np.random.seed(41)
    torch.manual_seed(41)
    torch.cuda.manual_seed_all(41)
    directory = RUNS_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    network = build_maskrcnn(size)
    optimizer = torch.optim.SGD(
        [p for p in network.parameters() if p.requires_grad],
        lr=0.005, momentum=0.9, weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 27], gamma=0.1)
    scaler = torch.amp.GradScaler("cuda")
    generator = torch.Generator().manual_seed(41)
    loader = DataLoader(
        train_dataset, batch_size=1, shuffle=True, num_workers=0,
        collate_fn=collate_detection, generator=generator,
    )
    start_epoch, best_ap, rows = 0, -1.0, []
    checkpoint = directory / "last.pth"
    # Memulihkan run parsial dari Output versi lama yang ditambahkan sebagai Input.
    if not checkpoint.exists():
        candidates = sorted(Path("/kaggle/input").glob(f"**/{name}/last.pth"))
        if candidates:
            checkpoint = candidates[0]
            if (checkpoint.parent / "best.pth").exists():
                shutil.copy2(checkpoint.parent / "best.pth", directory / "best.pth")
    if checkpoint.exists():
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
        network.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, best_ap, rows = state["epoch"], state["best_ap"], state["history"]
        generator.set_state(state["generator_state"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        del state
        print("Lanjut dari epoch:", start_epoch)

    config = {
        "model": "Mask R-CNN ResNet-50 FPN", "input_size": size,
        "resize": "longest edge with aspect ratio preserved",
        "epochs": 30, "seed": 41, "batch_size": 1, "gradient_accumulation": 4,
        "optimizer": "SGD", "learning_rate": 0.005, "milestones": [20, 27],
        "best_selection": "validation COCO mask AP50:95", "versions": versions,
    }
    (directory / "config.json").write_text(json.dumps(config, indent=2))
    (directory / "classes.json").write_text(json.dumps({
        "names": names, "foreground_offset": 1,
        "label_to_coco_category": train_dataset.label_to_category,
    }, indent=2))

    for epoch in range(start_epoch, 30):
        started = time.time()
        network.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        for step, (images, targets) in enumerate(tqdm(loader, desc=f"{name} epoch {epoch + 1}/30")):
            images = [image.cuda() for image in images]
            targets = [{key: value.cuda() for key, value in target.items()} for target in targets]
            # Kelompok terakhir boleh kurang dari empat gambar.
            group_size = min(4, len(loader) - (step // 4) * 4)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                losses = network(images, targets)
                loss = sum(losses.values())
            if not math.isfinite(float(loss)):
                raise RuntimeError(f"Loss tidak finite: epoch {epoch + 1}, step {step}")
            scaler.scale(loss / group_size).backward()
            if (step + 1) % 4 == 0 or step + 1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(network.parameters(), 10)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            total_loss += float(loss.detach())
        scheduler.step()
        ap, ap50 = validation_mask_ap(network)
        improved = ap > best_ap
        best_ap = max(best_ap, ap)
        rows.append({
            "epoch": epoch + 1, "train_loss": total_loss / len(loader),
            "val_mask_map50_95": ap, "val_mask_map50": ap50,
            "seconds": time.time() - started,
        })
        with (directory / "results.csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        state = {
            "epoch": epoch + 1, "model": network.state_dict(),
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "best_ap": best_ap, "history": rows,
            "config": config, "names": names, "generator_state": generator.get_state(),
            "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
            "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
        }
        temporary = directory / "last.tmp.pth"
        torch.save(state, temporary)
        temporary.replace(directory / "last.pth")
        if improved:
            shutil.copy2(directory / "last.pth", directory / "best.pth")
        del state
        print(rows[-1], flush=True)

    finish_run(directory, config)
    del network, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return directory


### Run

In [41]:
import pandas as pd
from IPython.display import display

training_summary = {}


def show_training_result(directory, model_name, size):
    directory = Path(directory)
    history = pd.read_csv(directory / "results.csv")
    history.columns = history.columns.str.strip()
    if model_name == "Mask R-CNN ResNet-50 FPN":
        metric_column = "val_mask_map50_95"
        checkpoint = directory / "best.pth"
    else:
        metric_column = "metrics/mAP50-95(M)"
        checkpoint = directory / "weights" / "best.pt"
    record = {
        "Model": model_name,
        "Input": size,
        "Epoch selesai": len(history),
        "Val mask AP50–95 terbaik": float(history[metric_column].max()),
        "Checkpoint": str(checkpoint),
    }
    training_summary[(model_name, size)] = record
    display(pd.DataFrame([record]).style.format({"Val mask AP50–95 terbaik": "{:.4f}"}))
    pd.DataFrame(training_summary.values()).to_csv(
        WORK_DIR / "ringkasan_training.csv", index=False
    )

## YOLOv8n-seg

### YOLOv8n-seg · input 640

In [42]:
if not EVALUATION_ONLY:
    run_yolov8n_seg_640 = train_yolo_run("yolov8n-seg.pt", 640, "yolov8n_seg_640_resmi")
    show_training_result(run_yolov8n_seg_640, "YOLOv8n-seg", 640)

### YOLOv8n-seg · input 320

In [43]:
if not EVALUATION_ONLY:
    run_yolov8n_seg_320 = train_yolo_run("yolov8n-seg.pt", 320, "yolov8n_seg_320_resmi")
    show_training_result(run_yolov8n_seg_320, "YOLOv8n-seg", 320)

### YOLOv8n-seg · input 960

In [44]:
if not EVALUATION_ONLY:
    run_yolov8n_seg_960 = train_yolo_run("yolov8n-seg.pt", 960, "yolov8n_seg_960_resmi")
    show_training_result(run_yolov8n_seg_960, "YOLOv8n-seg", 960)

## YOLOv8s-seg

### YOLOv8s-seg · input 640

In [45]:
if not EVALUATION_ONLY:
    run_yolov8s_seg_640 = train_yolo_run("yolov8s-seg.pt", 640, "yolov8s_seg_640_resmi")
    show_training_result(run_yolov8s_seg_640, "YOLOv8s-seg", 640)

### YOLOv8s-seg · input 320

In [46]:
if not EVALUATION_ONLY:
    run_yolov8s_seg_320 = train_yolo_run("yolov8s-seg.pt", 320, "yolov8s_seg_320_resmi")
    show_training_result(run_yolov8s_seg_320, "YOLOv8s-seg", 320)

### YOLOv8s-seg · input 960

In [47]:
if not EVALUATION_ONLY:
    run_yolov8s_seg_960 = train_yolo_run("yolov8s-seg.pt", 960, "yolov8s_seg_960_resmi")
    show_training_result(run_yolov8s_seg_960, "YOLOv8s-seg", 960)

## Mask R-CNN ResNet-50 FPN

In [48]:
import contextlib
import copy
import io
import math

from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
from tqdm.auto import tqdm


class IndoorCocoDataset(Dataset):
    def __init__(self, split, class_names):
        self.split = split
        self.image_dir = root / split
        source = json.loads((self.image_dir / "_annotations.coco.json").read_text())
        self.images = source["images"]
        self.foreground_id = {name: i + 1 for i, name in enumerate(class_names)}
        self.category_to_label = {
            cat["id"]: self.foreground_id[cat["name"]]
            for cat in source["categories"]
        }
        self.label_to_category = {value: key for key, value in self.category_to_label.items()}
        self.annotations = defaultdict(list)
        filtered = []
        seen = set()
        for ann in source["annotations"]:
            key = (ann["image_id"], ann["category_id"], json.dumps(ann["segmentation"]))
            if key in seen:
                continue
            seen.add(key)
            filtered.append(ann)
            self.annotations[ann["image_id"]].append(ann)

        source["annotations"] = filtered
        self.coco = COCO()
        self.coco.dataset = copy.deepcopy(source)
        self.coco.dataset.setdefault("info", {})
        self.coco.createIndex()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        info = self.images[index]
        with Image.open(self.image_dir / info["file_name"]) as image:
            image = to_tensor(image.convert("RGB"))
        height, width = image.shape[-2:]
        scale_x = width / info["width"]
        scale_y = height / info["height"]
        boxes, labels, masks = [], [], []
        for ann in self.annotations[info["id"]]:
            if ann.get("iscrowd"):
                continue
            mask = self.coco.annToMask(ann).astype(np.uint8)
            if mask.shape != (height, width):
                mask = np.asarray(Image.fromarray(mask).resize(
                    (width, height), resample=Image.Resampling.NEAREST
                )).copy()
            if not mask.any():
                raise ValueError(f"Mask kosong pada ID {ann['id']}")
            x, y, w, h = ann["bbox"]
            box = [
                max(0, x * scale_x), max(0, y * scale_y),
                min(width, (x + w) * scale_x), min(height, (y + h) * scale_y),
            ]
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError(f"BBox tidak valid pada ID {ann['id']}")
            boxes.append(box)
            labels.append(self.category_to_label[ann["category_id"]])
            masks.append(mask)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "masks": torch.from_numpy(np.stack(masks)) if masks else torch.zeros(
                (0, height, width), dtype=torch.uint8
            ),
            "image_id": torch.tensor(info["id"]),
        }
        return image, target


def collate_detection(batch):
    return tuple(zip(*batch))


train_dataset = IndoorCocoDataset("train", names)
valid_dataset = IndoorCocoDataset("valid", names)


def build_maskrcnn(size):
    network = maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,
        min_size=size, max_size=size,
        box_score_thresh=0.001, box_detections_per_img=100,
    )
    box_features = network.roi_heads.box_predictor.cls_score.in_features
    network.roi_heads.box_predictor = FastRCNNPredictor(box_features, len(names) + 1)
    mask_features = network.roi_heads.mask_predictor.conv5_mask.in_channels
    network.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, len(names) + 1)
    return network.cuda()


@torch.inference_mode()
def validation_mask_ap(network):
    network.eval()
    predictions = []
    for image, target in tqdm(valid_dataset, desc="Validasi mask", total=len(valid_dataset)):
        output = network([image.cuda()])[0]
        image_id = int(target["image_id"])
        info = valid_dataset.coco.imgs[image_id]
        for label, score, mask in zip(output["labels"], output["scores"], output["masks"]):
            # COCO mengevaluasi mask pada ukuran koordinat anotasi sumber.
            probability = F.interpolate(
                mask.unsqueeze(0), size=(info["height"], info["width"]),
                mode="bilinear", align_corners=False,
            )[0, 0]
            binary = (probability >= 0.5).cpu().numpy().astype(np.uint8)
            if not binary.any():
                continue
            encoded = mask_utils.encode(np.asfortranarray(binary))
            encoded["counts"] = encoded["counts"].decode("ascii")
            predictions.append({
                "image_id": image_id,
                "category_id": valid_dataset.label_to_category[int(label)],
                "segmentation": encoded,
                "score": float(score),
            })
        del output

    if not predictions:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        predicted_coco = valid_dataset.coco.loadRes(predictions)
        evaluation = COCOeval(valid_dataset.coco, predicted_coco, "segm")
        evaluation.params.imgIds = [image["id"] for image in valid_dataset.images]
        evaluation.evaluate()
        evaluation.accumulate()
        evaluation.summarize()
    return float(evaluation.stats[0]), float(evaluation.stats[1])

creating index...
index created!
creating index...
index created!


### Mask R-CNN ResNet-50 FPN · input 640

In [49]:
if not EVALUATION_ONLY:
    run_maskrcnn_r50_fpn_640 = train_maskrcnn_run(640)
    show_training_result(run_maskrcnn_r50_fpn_640, "Mask R-CNN ResNet-50 FPN", 640)

### Mask R-CNN ResNet-50 FPN · input 320

In [50]:
if not EVALUATION_ONLY:
    run_maskrcnn_r50_fpn_320 = train_maskrcnn_run(320)
    show_training_result(run_maskrcnn_r50_fpn_320, "Mask R-CNN ResNet-50 FPN", 320)

### Mask R-CNN ResNet-50 FPN · input 960

In [51]:
if not EVALUATION_ONLY:
    run_maskrcnn_r50_fpn_960 = train_maskrcnn_run(960)
    show_training_result(run_maskrcnn_r50_fpn_960, "Mask R-CNN ResNet-50 FPN", 960)

## Ringkasan sembilan eksperimen

In [52]:
if not EVALUATION_ONLY:
    summary = pd.DataFrame(training_summary.values())
    summary.to_csv(WORK_DIR / "ringkasan_training.csv", index=False)
    display(summary.style.format({"Val mask AP50–95 terbaik": "{:.4f}"}))
    print(f"Konfigurasi selesai: {len(summary)}/9")
    print("Ringkasan:", WORK_DIR / "ringkasan_training.csv")
    print("ZIP tersedia pada sesi ini:")
    for archive in sorted(WORK_DIR.glob("*.zip")):
        print(f"{archive.name} ({archive.stat().st_size / 1024**2:.1f} MB)")

# Evaluasi test resmi dan test sendiri

## Persiapan test set dan pemeriksaan sembilan checkpoint


In [53]:
import contextlib
import copy
import gc
import hashlib
import io
import json
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from kaggle_secrets import UserSecretsClient
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from roboflow import Roboflow
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm
from ultralytics import YOLO

EVAL_DIR = Path('/kaggle/working/evaluasi_041')
EVAL_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_NAMES = '''indoor-objects|backpack|bed|blanket|book|bottle|bowl|box|cabinet-merged|cardboard|ceiling-merged|cell phone|chair|clock|couch|counter|cup|curtain|dining table|door-stuff|fire-extinguisher|floor-other-merged|floor-wood|flower|food-other-merged|fork|hair drier|handbag|keyboard|knife|laptop|light|microwave|mirror-stuff|mouse|oven|pan|paper|paper-merged|person|pillow|plate|plug|potted plant|refrigerator|remote|rug-merged|scissor|scissors|shelf|sink|spoon|st-|stairs|stove-merged|table-merged|teddy bear|toaster|toilet|toothbrush|towel|toy|trash-bin|tree-merged|tv|vase|wall|wall-brick|wall-other-merged|wall-stone|wall-tile|wall-wood|wallet|window-blind|window-other'''.split('|')
CATEGORY_BY_NAME = {name: i + 1 for i, name in enumerate(CANONICAL_NAMES)}

# Boleh diisi manual jika pencarian otomatis menemukan lebih dari satu salinan.
SELF_COCO_PATH = ''
OFFICIAL_COCO_PATH = ''
CHECKPOINT_PATHS = {}
# Contoh CHECKPOINT_PATHS['yolov8n_seg_640_resmi'] = '/kaggle/input/.../weights/best.pt'


def locate_self_test():
    if SELF_COCO_PATH:
        return Path(SELF_COCO_PATH)
    candidates = list(Path('/kaggle/input').glob('**/_annotations.coco.json'))
    for candidate in candidates:
        data = json.loads(candidate.read_text())
        if len(data.get('images', [])) == 44 and len(data.get('annotations', [])) == 362:
            return candidate
    archives = list(Path('/kaggle/input').glob('**/Indoor_Test_Sendiri_041.zip'))
    if len(archives) != 1:
        raise FileNotFoundError('Tambahkan ZIP Indoor_Test_Sendiri_041.zip sebagai Input Kaggle, atau isi SELF_COCO_PATH.')
    destination = EVAL_DIR / 'test_sendiri'
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if member.is_dir() or '__MACOSX' in member.filename or Path(member.filename).name.startswith('._'):
                continue
            relative = Path(member.filename)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError('Path tidak aman dalam ZIP.')
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
    matches = list(destination.glob('**/_annotations.coco.json'))
    if len(matches) != 1:
        raise ValueError('Isi SELF_COCO_PATH dengan lokasi JSON test sendiri.')
    return matches[0]


def locate_official_test():
    if OFFICIAL_COCO_PATH:
        return Path(OFFICIAL_COCO_PATH)
    # Dataset resmi dari sel persiapan notebook training dapat dipakai kembali.
    if 'root' in globals():
        candidate = Path(root) / 'test' / '_annotations.coco.json'
        if candidate.exists():
            return candidate
    rf_eval = Roboflow(api_key=UserSecretsClient().get_secret('ROBOFLOW_API_KEY'))
    downloaded = rf_eval.workspace('panopticindoor').project('panoptic-indoor-segmentation').version(5).download(
        'coco-segmentation', location=str(EVAL_DIR / 'dataset_resmi_v5')
    )
    return Path(downloaded.location) / 'test' / '_annotations.coco.json'


def prepare_ground_truth(path, expected_images):
    source = json.loads(Path(path).read_text())
    data = copy.deepcopy(source)
    category_names = {cat['id']: cat['name'] for cat in source['categories']}
    used_ids = {ann['category_id'] for ann in source['annotations']}
    unknown = {category_names[key] for key in used_ids} - set(CATEGORY_BY_NAME)
    if unknown:
        raise ValueError(f'Nama kelas anotasi tidak sesuai training: {unknown}')
    if len(source['images']) != expected_images:
        raise ValueError(f'Diharapkan {expected_images} gambar, ditemukan {len(source["images"])}.')
    annotations, seen = [], set()
    for ann in source['annotations']:
        category = CATEGORY_BY_NAME[category_names[ann['category_id']]]
        key = (ann['image_id'], category, json.dumps(ann['segmentation'], sort_keys=True))
        if key in seen:
            continue
        seen.add(key)
        annotations.append({**ann, 'id': len(annotations) + 1, 'category_id': category, 'iscrowd': ann.get('iscrowd', 0)})
    data['annotations'] = annotations
    data['categories'] = [{'id': key, 'name': name} for name, key in CATEGORY_BY_NAME.items()]
    data.setdefault('info', {})
    for image in data['images']:
        if not (Path(path).parent / image['file_name']).is_file():
            raise FileNotFoundError(image['file_name'])
    coco = COCO()
    coco.dataset = data
    coco.createIndex()
    print(Path(path).parent, '| gambar', len(data['images']), '| mask', len(annotations))
    return {'path': Path(path), 'coco': coco, 'data': data}


self_path = locate_self_test()
official_path = locate_official_test()
EVAL_DATASETS = {
    'resmi': prepare_ground_truth(official_path, 359),
    'sendiri': prepare_ground_truth(self_path, 44),
}
SHARED_CATEGORIES = sorted(set.intersection(*[
    {ann['category_id'] for ann in item['data']['annotations'] if not ann.get('iscrowd')}
    for item in EVAL_DATASETS.values()
]))
print('Jumlah kelas yang mempunyai anotasi di kedua test set:', len(SHARED_CATEGORIES))
assert torch.cuda.is_available(), 'Aktifkan GPU Kaggle untuk evaluasi.'


def locate_checkpoint(name, family):
    if name in CHECKPOINT_PATHS:
        path = Path(CHECKPOINT_PATHS[name])
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    filename = 'best.pt' if family == 'yolo' else 'best.pth'
    candidates = []
    for base in (Path('/kaggle/working/visi_komputer_041'), Path('/kaggle/input')):
        for path in base.glob(f'**/{filename}'):
            if name in path.parts:
                candidates.append(path)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError(f'Ada beberapa checkpoint {name}. Isi CHECKPOINT_PATHS. Ditemukan: {candidates}')
    archives = list(Path('/kaggle/input').glob(f'**/{name}.zip'))
    if len(archives) != 1:
        raise FileNotFoundError(f'Checkpoint {name} belum tersedia. Tambahkan Output notebook training sebagai Input Kaggle.')
    member_name = 'weights/best.pt' if family == 'yolo' else 'best.pth'
    destination = EVAL_DIR / 'checkpoints' / name / member_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        destination.write_bytes(archive.read(member_name))
    return destination


RUN_CONFIGS = [
    (f'{slug}_{size}_resmi', family, label, size)
    for slug, family, label in (
        ('yolov8n_seg', 'yolo', 'YOLOv8n-seg'),
        ('yolov8s_seg', 'yolo', 'YOLOv8s-seg'),
        ('maskrcnn_r50_fpn', 'maskrcnn', 'Mask R-CNN ResNet-50 FPN'),
    )
    for size in (640, 320, 960)
]
EVAL_CHECKPOINTS = {name: locate_checkpoint(name, family) for name, family, _, _ in RUN_CONFIGS}
display(pd.DataFrame([{'Run': name, 'Checkpoint': str(path)} for name, path in EVAL_CHECKPOINTS.items()]))

creating index...
index created!
/kaggle/working/panoptic_indoor_coco_v5/test | gambar 359 | mask 5341
creating index...
index created!
/kaggle/input/datasets/sittiaminah/indoor-test-sendiri-041/Indoor_Test_Sendiri_041/test | gambar 44 | mask 362
Jumlah kelas yang mempunyai anotasi di kedua test set: 49


,Run,Checkpoint
0,yolov8n_seg_640_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
1,yolov8n_seg_320_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
2,yolov8n_seg_960_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
3,yolov8s_seg_640_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
4,yolov8s_seg_320_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
5,yolov8s_seg_960_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
6,maskrcnn_r50_fpn_640_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
7,maskrcnn_r50_fpn_320_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...
8,maskrcnn_r50_fpn_960_resmi,/kaggle/input/notebooks/sittiaminah/viskom-tra...


## Fungsi evaluasi COCO mask


In [54]:
def encode_mask(binary):
    encoded = mask_utils.encode(np.asfortranarray(binary.astype(np.uint8)))
    encoded['counts'] = encoded['counts'].decode('ascii')
    return encoded


def load_evaluation_model(family, checkpoint, size):
    if family == 'yolo':
        model = YOLO(str(checkpoint))
        if [model.names[i] for i in range(len(model.names))] != CANONICAL_NAMES:
            raise ValueError('Urutan kelas checkpoint YOLO berbeda dari daftar training.')
        return model
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    if state['names'] != CANONICAL_NAMES or state['config']['input_size'] != size:
        raise ValueError('Kelas atau ukuran checkpoint Mask R-CNN tidak cocok.')
    model = maskrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, min_size=size, max_size=size,
        box_score_thresh=0.001, box_detections_per_img=100,
    )
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, len(CANONICAL_NAMES) + 1)
    features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(features, 256, len(CANONICAL_NAMES) + 1)
    model.load_state_dict(state['model'])
    del state
    return model.cuda().eval()


@torch.inference_mode()
def predict_test(model, family, size, dataset):
    records = []
    started = time.perf_counter()
    for info in tqdm(dataset['data']['images'], desc=f'Evaluasi {family} {size}'):
        path = dataset['path'].parent / info['file_name']
        height, width = info['height'], info['width']
        if family == 'yolo':
            result = model.predict(
                source=str(path), imgsz=size, conf=0.001, iou=0.7,
                max_det=100, device=0, verbose=False, retina_masks=False,
            )[0]
            if result.masks is None:
                continue
            actual_height, actual_width = result.orig_shape
            for box, polygon in zip(result.boxes, result.masks.xy):
                if len(polygon) < 3:
                    continue
                polygon = polygon.astype(float) * [width / actual_width, height / actual_height]
                rles = mask_utils.frPyObjects([polygon.reshape(-1).tolist()], height, width)
                encoded = mask_utils.merge(rles)
                encoded['counts'] = encoded['counts'].decode('ascii')
                category = int(box.cls.item()) + 1
                score = float(box.conf.item())
                records.append({'image_id': info['id'], 'category_id': category, 'segmentation': encoded, 'score': score})
            del result
        else:
            # Resize sebelum inference membatasi alokasi mask untuk foto HP beresolusi tinggi.
            with Image.open(path) as image:
                image = image.convert('RGB')
                scale = size / max(image.size)
                image = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.Resampling.BILINEAR)
                tensor = to_tensor(image).cuda()
            output = model([tensor])[0]
            for label, score, probability in zip(output['labels'], output['scores'], output['masks']):
                probability = Image.fromarray(probability[0].cpu().numpy()).resize((width, height), Image.Resampling.BILINEAR)
                binary = np.asarray(probability) >= 0.5
                if not binary.any():
                    continue
                records.append({
                    'image_id': info['id'], 'category_id': int(label.item()),
                    'segmentation': encode_mask(binary), 'score': float(score.item()),
                })
            del output, tensor
    torch.cuda.synchronize()
    return records, time.perf_counter() - started


def score_coco(dataset, records, category_ids):
    gt = dataset['coco']
    with contextlib.redirect_stdout(io.StringIO()):
        if records:
            clean_records = [
                {
                    "image_id": int(item["image_id"]),
                    "category_id": int(item["category_id"]),
                    "score": float(item["score"]),
                    "segmentation": copy.deepcopy(item["segmentation"]),
                }
                for item in records
            ]
            predictions = gt.loadRes(clean_records)
        else:
            predictions = COCO()
            predictions.dataset = {**copy.deepcopy(gt.dataset), 'annotations': []}
            predictions.createIndex()
        evaluator = COCOeval(gt, predictions, 'segm')
        evaluator.params.imgIds = [image['id'] for image in dataset['data']['images']]
        evaluator.params.catIds = category_ids
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    per_class = []
    for index, category_id in enumerate(evaluator.params.catIds):
        values = evaluator.eval['precision'][:, :, index, 0, -1]
        values = values[values >= 0]
        if len(values):
            per_class.append({'class': CANONICAL_NAMES[category_id - 1], 'mask_AP50_95': float(values.mean())})
    return {'mask_mAP50_95': float(evaluator.stats[0]), 'mask_mAP50': float(evaluator.stats[1]), 'mask_AR100': float(evaluator.stats[8])}, per_class


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def evaluate_configuration(name):
    config = next(item for item in RUN_CONFIGS if item[0] == name)
    _, family, label, size = config
    checkpoint = EVAL_CHECKPOINTS[name]
    checkpoint_hash = file_digest(checkpoint)
    model = None
    output_rows = []
    for split, dataset in EVAL_DATASETS.items():
        folder = EVAL_DIR / name / split
        folder.mkdir(parents=True, exist_ok=True)
        signature = {
            'checkpoint_sha256': checkpoint_hash, 'ground_truth_sha256': file_digest(dataset['path']),
            'size': size, 'confidence': 0.001, 'max_detections': 100, 'protocol': 'coco_segm_v1',
        }
        prediction_path, cache_path = folder / 'predictions.json', folder / 'cache.json'
        if prediction_path.exists() and cache_path.exists() and json.loads(cache_path.read_text())['signature'] == signature:
            cache = json.loads(cache_path.read_text())
            predictions = json.loads(prediction_path.read_text())
            seconds = cache['seconds']
        else:
            if model is None:
                model = load_evaluation_model(family, checkpoint, size)
            predictions, seconds = predict_test(model, family, size, dataset)
            prediction_path.write_text(json.dumps(predictions))
            cache_path.write_text(json.dumps({'signature': signature, 'seconds': seconds}))
        for scope, category_ids in [('all', list(CATEGORY_BY_NAME.values())), ('shared', SHARED_CATEGORIES)]:
            metrics, per_class = score_coco(dataset, predictions, category_ids)
            pd.DataFrame(per_class).to_csv(folder / f'per_class_{scope}.csv', index=False)
            output_rows.append({
                'run': name, 'model': label, 'input': size, 'test_set': split,
                'class_scope': scope, 'images': len(dataset['data']['images']),
                **metrics, 'pipeline_ms_per_image': seconds * 1000 / len(dataset['data']['images']),
            })
        pd.DataFrame(output_rows).to_csv(EVAL_DIR / name / 'metrics.csv', index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    display(pd.DataFrame(output_rows))
    return output_rows

## Evaluasi YOLOv8n-seg


### Input 640 · test resmi dan test sendiri


In [55]:
eval_yolov8n_seg_640 = evaluate_configuration("yolov8n_seg_640_resmi")

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8n_seg_640_resmi,YOLOv8n-seg,640,resmi,all,359,0.063393,0.107739,0.180886,118.709440
1,yolov8n_seg_640_resmi,YOLOv8n-seg,640,resmi,shared,359,0.077811,0.134733,0.204908,118.709440
2,yolov8n_seg_640_resmi,YOLOv8n-seg,640,sendiri,all,44,0.117889,0.163751,0.241635,620.479347
3,yolov8n_seg_640_resmi,YOLOv8n-seg,640,sendiri,shared,44,0.125107,0.173776,0.256429,620.479347


### Input 320 · test resmi dan test sendiri


In [56]:
eval_yolov8n_seg_320 = evaluate_configuration("yolov8n_seg_320_resmi")

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8n_seg_320_resmi,YOLOv8n-seg,320,resmi,all,359,0.040024,0.073226,0.126599,49.991094
1,yolov8n_seg_320_resmi,YOLOv8n-seg,320,resmi,shared,359,0.040173,0.079096,0.130567,49.991094
2,yolov8n_seg_320_resmi,YOLOv8n-seg,320,sendiri,all,44,0.084325,0.142124,0.151326,229.770120
3,yolov8n_seg_320_resmi,YOLOv8n-seg,320,sendiri,shared,44,0.089488,0.150826,0.160591,229.770120


### Input 960 · test resmi dan test sendiri


In [57]:
eval_yolov8n_seg_960 = evaluate_configuration("yolov8n_seg_960_resmi")

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8n_seg_960_resmi,YOLOv8n-seg,960,resmi,all,359,0.060576,0.099758,0.183865,135.948518
1,yolov8n_seg_960_resmi,YOLOv8n-seg,960,resmi,shared,359,0.075700,0.125406,0.209891,135.948518
2,yolov8n_seg_960_resmi,YOLOv8n-seg,960,sendiri,all,44,0.099115,0.140138,0.235043,337.033022
3,yolov8n_seg_960_resmi,YOLOv8n-seg,960,sendiri,shared,44,0.101515,0.142757,0.237189,337.033022


## Evaluasi YOLOv8s-seg


### Input 640 · test resmi dan test sendiri


In [58]:
eval_yolov8s_seg_640 = evaluate_configuration("yolov8s_seg_640_resmi")

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8s_seg_640_resmi,YOLOv8s-seg,640,resmi,all,359,0.100488,0.155379,0.239290,88.386527
1,yolov8s_seg_640_resmi,YOLOv8s-seg,640,resmi,shared,359,0.110590,0.176317,0.272619,88.386527
2,yolov8s_seg_640_resmi,YOLOv8s-seg,640,sendiri,all,44,0.160686,0.231140,0.284923,277.679512
3,yolov8s_seg_640_resmi,YOLOv8s-seg,640,sendiri,shared,44,0.169905,0.244261,0.296245,277.679512


### Input 320 · test resmi dan test sendiri


In [59]:
eval_yolov8s_seg_320 = evaluate_configuration("yolov8s_seg_320_resmi")

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8s_seg_320_resmi,YOLOv8s-seg,320,resmi,all,359,0.064965,0.108102,0.174597,50.336197
1,yolov8s_seg_320_resmi,YOLOv8s-seg,320,resmi,shared,359,0.066543,0.119305,0.194681,50.336197
2,yolov8s_seg_320_resmi,YOLOv8s-seg,320,sendiri,all,44,0.121250,0.189015,0.208676,214.681372
3,yolov8s_seg_320_resmi,YOLOv8s-seg,320,sendiri,shared,44,0.128674,0.200587,0.221452,214.681372


### Input 960 · test resmi dan test sendiri


In [60]:
eval_yolov8s_seg_960 = evaluate_configuration("yolov8s_seg_960_resmi")

Evaluasi yolo 960:   0%|          | 0/44 [00:00<?, ?it/s]

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8s_seg_960_resmi,YOLOv8s-seg,960,resmi,all,359,0.105265,0.156696,0.237147,119.611163
1,yolov8s_seg_960_resmi,YOLOv8s-seg,960,resmi,shared,359,0.116663,0.179595,0.268042,119.611163
2,yolov8s_seg_960_resmi,YOLOv8s-seg,960,sendiri,all,44,0.136575,0.201278,0.230119,326.763209
3,yolov8s_seg_960_resmi,YOLOv8s-seg,960,sendiri,shared,44,0.137723,0.203296,0.237065,326.763209


## Evaluasi Mask R-CNN ResNet-50 FPN


### Input 640 · test resmi dan test sendiri


In [61]:
eval_maskrcnn_r50_fpn_640 = evaluate_configuration("maskrcnn_r50_fpn_640_resmi")

Evaluasi maskrcnn 640:   0%|          | 0/359 [00:00<?, ?it/s]

Evaluasi maskrcnn 640:   0%|          | 0/44 [00:00<?, ?it/s]

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,maskrcnn_r50_fpn_640_resmi,Mask R-CNN ResNet-50 FPN,640,resmi,all,359,0.080198,0.135245,0.222773,819.634698
1,maskrcnn_r50_fpn_640_resmi,Mask R-CNN ResNet-50 FPN,640,resmi,shared,359,0.080610,0.141200,0.240720,819.634698
2,maskrcnn_r50_fpn_640_resmi,Mask R-CNN ResNet-50 FPN,640,sendiri,all,44,0.144245,0.226382,0.294764,25385.992590
3,maskrcnn_r50_fpn_640_resmi,Mask R-CNN ResNet-50 FPN,640,sendiri,shared,44,0.150407,0.236223,0.299545,25385.992590


### Input 320 · test resmi dan test sendiri


In [62]:
eval_maskrcnn_r50_fpn_320 = evaluate_configuration("maskrcnn_r50_fpn_320_resmi")

Evaluasi maskrcnn 320:   0%|          | 0/359 [00:00<?, ?it/s]

Evaluasi maskrcnn 320:   0%|          | 0/44 [00:00<?, ?it/s]

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,maskrcnn_r50_fpn_320_resmi,Mask R-CNN ResNet-50 FPN,320,resmi,all,359,0.054964,0.102447,0.171001,696.978863
1,maskrcnn_r50_fpn_320_resmi,Mask R-CNN ResNet-50 FPN,320,resmi,shared,359,0.051480,0.102089,0.191337,696.978863
2,maskrcnn_r50_fpn_320_resmi,Mask R-CNN ResNet-50 FPN,320,sendiri,all,44,0.110930,0.179465,0.252613,25571.351082
3,maskrcnn_r50_fpn_320_resmi,Mask R-CNN ResNet-50 FPN,320,sendiri,shared,44,0.117436,0.190023,0.260936,25571.351082


### Input 960 · test resmi dan test sendiri


In [63]:
eval_maskrcnn_r50_fpn_960 = evaluate_configuration("maskrcnn_r50_fpn_960_resmi")

Evaluasi maskrcnn 960:   0%|          | 0/359 [00:00<?, ?it/s]

Evaluasi maskrcnn 960:   0%|          | 0/44 [00:00<?, ?it/s]

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,maskrcnn_r50_fpn_960_resmi,Mask R-CNN ResNet-50 FPN,960,resmi,all,359,0.086147,0.147844,0.233320,1071.318203
1,maskrcnn_r50_fpn_960_resmi,Mask R-CNN ResNet-50 FPN,960,resmi,shared,359,0.086610,0.152084,0.253620,1071.318203
2,maskrcnn_r50_fpn_960_resmi,Mask R-CNN ResNet-50 FPN,960,sendiri,all,44,0.165609,0.244746,0.338685,28411.719197
3,maskrcnn_r50_fpn_960_resmi,Mask R-CNN ResNet-50 FPN,960,sendiri,shared,44,0.172501,0.255173,0.344115,28411.719197


## Visualisasi

In [64]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Pilihan awal untuk ilustrasi. Bisa diganti dengan salah satu nama run lain.
VISUAL_RUN = 'yolov8s_seg_640_resmi'
VISUAL_CONF = 0.25
MATCH_IOU = 0.50
NUMBER_OF_EXAMPLES = 5
VISUAL_DIR = EVAL_DIR / 'perbandingan_visual' / VISUAL_RUN
VISUAL_DIR.mkdir(parents=True, exist_ok=True)

# Threshold ini hanya untuk gambar dan analisis kasus.
# Metrik AP sebelumnya tetap memakai confidence awal 0.001.
self_test = EVAL_DATASETS['sendiri']
gt_coco = self_test['coco']
prediction_file = EVAL_DIR / VISUAL_RUN / 'sendiri' / 'predictions.json'
if not prediction_file.is_file():
    raise FileNotFoundError(f'Jalankan sel evaluasi {VISUAL_RUN} sebelum visualisasi.')
visual_predictions = [
    item for item in json.loads(prediction_file.read_text())
    if item['score'] >= VISUAL_CONF
]
predictions_by_image = {}
for item in visual_predictions:
    predictions_by_image.setdefault(item['image_id'], []).append(item)


def mask_as_rle(coco, annotation):
    return coco.annToRLE(annotation)


def review_image(info):
    ground_truth = [
        item for item in gt_coco.imgToAnns[info['id']]
        if not item.get('iscrowd')
    ]
    predictions = sorted(
        predictions_by_image.get(info['id'], []),
        key=lambda item: item['score'], reverse=True,
    )
    matched_gt = set()
    matched_pred = set()
    if ground_truth and predictions:
        overlaps = mask_utils.iou(
            [item['segmentation'] for item in predictions],
            [mask_as_rle(gt_coco, item) for item in ground_truth],
            [0] * len(ground_truth),
        )
        # Greedy matching berdasarkan confidence, dengan syarat kelas sama.
        for pred_index, prediction in enumerate(predictions):
            available = [
                index for index, annotation in enumerate(ground_truth)
                if index not in matched_gt
                and annotation['category_id'] == prediction['category_id']
            ]
            if not available:
                continue
            best = max(available, key=lambda index: overlaps[pred_index, index])
            if overlaps[pred_index, best] >= MATCH_IOU:
                matched_gt.add(best)
                matched_pred.add(pred_index)
    missed = len(ground_truth) - len(matched_gt)
    extra = len(predictions) - len(matched_pred)
    return {
        'image_id': info['id'], 'file_name': info['file_name'],
        'ground_truth_objects': len(ground_truth), 'predicted_objects': len(predictions),
        'matched_objects': len(matched_gt), 'missed_objects': missed,
        'unmatched_predictions': extra,
        'error_fraction': (missed + extra) / max(1, len(ground_truth) + len(predictions)),
    }


review_table = pd.DataFrame([review_image(info) for info in self_test['data']['images']])
review_table = review_table.sort_values(
    ['error_fraction', 'missed_objects', 'unmatched_predictions'], ascending=False
)
review_table.to_csv(VISUAL_DIR / 'diagnostik_semua_foto.csv', index=False)
failure_examples = review_table[
    (review_table.missed_objects > 0) | (review_table.unmatched_predictions > 0)
].head(NUMBER_OF_EXAMPLES)
failure_examples.to_csv(VISUAL_DIR / 'kandidat_5_kegagalan.csv', index=False)
display(failure_examples)
if len(failure_examples) < NUMBER_OF_EXAMPLES:
    print('Jumlah kandidat kegagalan pada threshold ini:', len(failure_examples))


def overlay_annotations(axis, image, annotations, predictions=False):
    axis.imshow(image)
    displayed_classes = set()
    for annotation in annotations:
        if predictions:
            encoded = annotation['segmentation']
        else:
            encoded = mask_as_rle(gt_coco, annotation)
        mask = mask_utils.decode(encoded)
        if mask.ndim == 3:
            mask = mask.any(axis=2)
        resized = np.asarray(Image.fromarray(mask.astype(np.uint8)).resize(
            image.size, Image.Resampling.NEAREST
        )).astype(bool)
        if not resized.any():
            continue
        category = annotation['category_id']
        color = plt.get_cmap('tab20')((category - 1) % 20)
        layer = np.zeros((image.height, image.width, 4), dtype=np.float32)
        layer[resized] = (*color[:3], 0.35)
        axis.imshow(layer)
        if resized.any() and not resized.all():
            axis.contour(resized.astype(float), levels=[0.5], colors=[color], linewidths=0.7)
        yy, xx = np.where(resized)
        label = CANONICAL_NAMES[category - 1]
        if predictions:
            label += f" {annotation['score']:.2f}"
        axis.text(
            int(xx.min()), int(yy.min()), label, fontsize=6, color='white',
            bbox={'facecolor': color, 'alpha': 0.85, 'pad': 1},
        )
        displayed_classes.add(category)
    axis.axis('off')


for number, record in enumerate(failure_examples.to_dict('records'), start=1):
    info = gt_coco.imgs[record['image_id']]
    with Image.open(self_test['path'].parent / info['file_name']) as source:
        image = source.convert('RGB')
        image.thumbnail((900, 900))
    ground_truth = [item for item in gt_coco.imgToAnns[info['id']] if not item.get('iscrowd')]
    predictions = predictions_by_image.get(info['id'], [])
    figure, axes = plt.subplots(1, 3, figsize=(15, 7))
    axes[0].imshow(image)
    axes[0].axis('off')
    axes[0].set_title('Foto asli')
    overlay_annotations(axes[1], image, ground_truth)
    axes[1].set_title(f'Ground truth · {len(ground_truth)} objek')
    overlay_annotations(axes[2], image, predictions, predictions=True)
    axes[2].set_title(f'Prediksi · confidence ≥ {VISUAL_CONF}')
    figure.suptitle(
        f"{VISUAL_RUN} | contoh {number}\n"
        f"Tidak cocok dengan GT: {record['missed_objects']} objek | "
        f"Prediksi tanpa pasangan: {record['unmatched_predictions']} | IoU ≥ {MATCH_IOU}"
    )
    figure.tight_layout()
    figure.savefig(VISUAL_DIR / f'kasus_{number:02d}.png', dpi=160, bbox_inches='tight')
    plt.close(figure)

print('Tabel diagnostik dan gambar tersimpan di:', VISUAL_DIR)

,image_id,file_name,ground_truth_objects,predicted_objects,matched_objects,missed_objects,unmatched_predictions,error_fraction
7,7,IMG_1124_HEIC.rf.86859c386c74e7d151dae7dba07a5...,12,2,0,12,2,1.0
14,14,IMG_1116_HEIC.rf.02690352032886a6c2d8ab2267de1...,6,9,0,6,9,1.0
5,5,IMG_0984_HEIC.rf.9e0a35f45a085523ae53b0c663020...,5,5,0,5,5,1.0
2,2,IMG_1123_HEIC.rf.f39ca2cdef67f662cf8a83731dbf6...,4,3,0,4,3,1.0
41,41,IMG_1101_HEIC.rf.dbffeefde7db1a4cfa4cbf5023bae...,3,5,0,3,5,1.0


Tabel diagnostik dan gambar tersimpan di: /kaggle/working/evaluasi_041/perbandingan_visual/yolov8s_seg_640_resmi


## Perbandingan untuk 1 foto

In [65]:
import base64
import html
from PIL import ImageDraw

DEMO_DIR = EVAL_DIR / 'demo_dan_laporan'
DEMO_DIR.mkdir(parents=True, exist_ok=True)
CASE_IDS = failure_examples['image_id'].tolist()
if not CASE_IDS:
    CASE_IDS = [item['id'] for item in self_test['data']['images'][:5]]

# Lima foto yang sama dipakai untuk seluruh model agar perbandingan langsung.
# Pemilihannya berasal dari diagnostik model acuan VISUAL_RUN.
case_labels = [f"Foto {index + 1} · {gt_coco.imgs[image_id]['file_name'].split('.rf.')[0]}" for index, image_id in enumerate(CASE_IDS)]
image_assets = {}
diagnostic_rows = []
for run_name, family, label, input_size in RUN_CONFIGS:
    predictions = json.loads((EVAL_DIR / run_name / 'sendiri' / 'predictions.json').read_text())
    predictions_by_image = {}
    for prediction in predictions:
        if prediction['score'] >= VISUAL_CONF:
            predictions_by_image.setdefault(prediction['image_id'], []).append(prediction)
    for case_index, image_id in enumerate(CASE_IDS):
        info = gt_coco.imgs[image_id]
        diagnostic = review_image(info)
        diagnostic_rows.append({'run': run_name, 'case': case_index + 1, **diagnostic})
        with Image.open(self_test['path'].parent / info['file_name']) as source:
            image = source.convert('RGB')
            image.thumbnail((650, 650))
        ground_truth = [a for a in gt_coco.imgToAnns[image_id] if not a.get('iscrowd')]
        predicted = predictions_by_image.get(image_id, [])
        figure, axes = plt.subplots(1, 3, figsize=(12, 6))
        axes[0].imshow(image)
        axes[0].axis('off')
        axes[0].set_title('Foto asli')
        overlay_annotations(axes[1], image, ground_truth)
        axes[1].set_title('Ground truth')
        overlay_annotations(axes[2], image, predicted, predictions=True)
        axes[2].set_title(f'Prediksi · confidence ≥ {VISUAL_CONF}')
        figure.suptitle(f'{label} · input {input_size} · foto {case_index + 1}')
        figure.tight_layout()
        png_path = DEMO_DIR / f'{run_name}_foto_{case_index + 1}.png'
        figure.savefig(png_path, dpi=150, bbox_inches='tight')
        # JPEG terkompresi untuk demo offline yang ringkas.
        buffer = io.BytesIO()
        figure.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
        buffer.seek(0)
        jpeg = io.BytesIO()
        with Image.open(buffer) as rendered:
            rendered.convert('RGB').save(jpeg, format='JPEG', quality=82)
        image_assets[f'{run_name}|{case_index}'] = base64.b64encode(jpeg.getvalue()).decode('ascii')
        plt.close(figure)

pd.DataFrame(diagnostic_rows).to_csv(DEMO_DIR / 'diagnostik_9_model_5_foto.csv', index=False)
pd.DataFrame([{'foto': index + 1, **gt_coco.imgs[image_id]} for index, image_id in enumerate(CASE_IDS)]).to_csv(
    DEMO_DIR / 'daftar_5_foto.csv', index=False
)

# Grafik memakai semua kelas dan kedua test set.
all_metrics = pd.concat([
    pd.read_csv(EVAL_DIR / name / 'metrics.csv') for name, _, _, _ in RUN_CONFIGS
], ignore_index=True)
all_metrics.to_csv(DEMO_DIR / 'metrik_lengkap.csv', index=False)
main_metrics = all_metrics[all_metrics.class_scope == 'all']
shared_metrics = all_metrics[all_metrics.class_scope == 'shared']
labels = [f'{label}\n{size}' for _, _, label, size in RUN_CONFIGS]
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(RUN_CONFIGS))
for offset, split, color in [(-0.19, 'resmi', '#285a8e'), (0.19, 'sendiri', '#c77735')]:
    scores = [float(main_metrics[(main_metrics.run == name) & (main_metrics.test_set == split)].iloc[0].mask_mAP50_95) for name, _, _, _ in RUN_CONFIGS]
    bars = ax.bar(x + offset, scores, width=0.38, label=f'Test {split}', color=color)
    ax.bar_label(bars, fmt='%.3f', fontsize=7, padding=3)
ax.set_xticks(x, labels, fontsize=8)
ax.set_ylabel('Mask AP50–95')
ax.set_ylim(0, 1)
ax.set_title('Evaluasi sembilan konfigurasi · kelas beranotasi pada masing-masing test set')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(DEMO_DIR / 'perbandingan_mask_AP.png', dpi=200)
plt.close(fig)

comparison = shared_metrics.pivot(index=['model', 'input'], columns='test_set', values='mask_mAP50_95').reset_index()
comparison['delta_sendiri_minus_resmi'] = comparison['sendiri'] - comparison['resmi']
comparison.to_csv(DEMO_DIR / 'domain_shift_shared.csv', index=False)
fig, ax = plt.subplots(figsize=(10, 5))
row_labels = comparison['model'] + ' · ' + comparison['input'].astype(str)
colors = ['#285a8e' if value >= 0 else '#c77735' for value in comparison.delta_sendiri_minus_resmi]
ax.barh(row_labels, comparison.delta_sendiri_minus_resmi, color=colors)
ax.axvline(0, color='#555555', linewidth=0.8)
ax.set_xlabel('AP test sendiri − AP test resmi · kelas yang sama')
ax.set_title('Perubahan performa pada foto sendiri')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(DEMO_DIR / 'domain_shift_shared.png', dpi=200)
plt.close(fig)

# Buat lembar perbandingan 3×3 untuk setiap foto, masing-masing satu prediksi per konfigurasi.
for case_index, image_id in enumerate(CASE_IDS):
    info = gt_coco.imgs[image_id]
    with Image.open(self_test['path'].parent / info['file_name']) as source:
        image = source.convert('RGB')
        image.thumbnail((450, 450))
    fig, axes = plt.subplots(3, 3, figsize=(12, 13))
    for ax, (run_name, _, label, input_size) in zip(axes.ravel(), RUN_CONFIGS):
        records = json.loads((EVAL_DIR / run_name / 'sendiri' / 'predictions.json').read_text())
        chosen = [p for p in records if p['image_id'] == image_id and p['score'] >= VISUAL_CONF]
        overlay_annotations(ax, image, chosen, predictions=True)
        ax.set_title(f'{label} · {input_size}', fontsize=9)
    fig.suptitle(f'Foto {case_index + 1} · sembilan konfigurasi · confidence ≥ {VISUAL_CONF}')
    fig.tight_layout()
    fig.savefig(DEMO_DIR / f'semua_model_foto_{case_index + 1}.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

# HTML
options = ''.join(f'<option value="{name}">{html.escape(label)} · {size}</option>' for name, _, label, size in RUN_CONFIGS)
cases = ''.join(f'<option value="{index}">{html.escape(label)}</option>' for index, label in enumerate(case_labels))
metric_view = main_metrics[['model', 'input', 'test_set', 'mask_mAP50_95', 'mask_mAP50', 'mask_AR100']].copy()
table_html = metric_view.to_html(index=False, float_format=lambda value: f'{value:.4f}', border=0)
page = '''<!doctype html><html lang="id"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Evaluasi Instance Segmentation · NRP 041</title>
<style>body{font-family:Arial,sans-serif;background:#f5f6f8;color:#172535;margin:0}main{max-width:1250px;margin:auto;padding:28px}h1{font-size:28px;margin-bottom:8px}p{line-height:1.5;color:#526172}section{background:white;border:1px solid #dde2e8;border-radius:12px;padding:20px;margin:20px 0}.controls{display:flex;gap:20px;flex-wrap:wrap}label{display:flex;flex-direction:column;gap:8px;font-weight:600}select{font:inherit;padding:10px;max-width:100%;background:white;border:1px solid #bec8d4;border-radius:6px}img{width:100%;height:auto}table{width:100%;border-collapse:collapse;font-size:14px}td,th{text-align:left;padding:10px;border-bottom:1px solid #e5e9ef}.scroll{overflow-x:auto}summary{cursor:pointer;font-size:18px;font-weight:600}</style>
<main><h1>Evaluasi Instance Segmentation</h1><p>NRP 041 · 9 konfigurasi · test resmi dan 44 foto sendiri</p>
<section><div class="controls"><label>Model dan input<select id="model">__OPTIONS__</select></label><label>Foto<select id="case">__CASES__</select></label></div>
<p>Lima foto yang sama untuk semua model. Foto dipilih dari kandidat kegagalan model acuan __BASELINE__. Confidence tampilan __CONF__; pencocokan diagnostik memakai mask IoU __IOU__.</p><img id="comparison" alt="Foto asli, ground truth, dan mask prediksi"></section>
<section><details><summary>Tabel metrik dua test set</summary><p>Mask AP memakai confidence awal 0.001 dan maksimal 100 deteksi per gambar. Kelas beranotasi dapat berbeda antar-test set.</p><div class="scroll">__TABLE__</div></details></section>
<p>Anotasi ground truth diperiksa manusia. Kandidat kegagalan perlu ditinjau untuk menjelaskan mekanisme model dan merancang eksperimen pembuktian.</p></main>
<script>const assets=__ASSETS__;const model=document.getElementById('model');const photo=document.getElementById('case');function show(){document.getElementById('comparison').src='data:image/jpeg;base64,'+assets[model.value+'|'+photo.value]}model.addEventListener('change',show);photo.addEventListener('change',show);show();</script></html>'''
page = page.replace('__OPTIONS__', options).replace('__CASES__', cases).replace('__TABLE__', table_html)
page = page.replace('__BASELINE__', html.escape(VISUAL_RUN)).replace('__CONF__', str(VISUAL_CONF)).replace('__IOU__', str(MATCH_IOU))
page = page.replace('__ASSETS__', json.dumps(image_assets))
demo_path = DEMO_DIR / 'demo_evaluasi_041.html'
demo_path.write_text(page, encoding='utf-8')
print('Demo offline:', demo_path)
print('Gambar laporan dan CSV:', DEMO_DIR)
from IPython.display import FileLink
display(FileLink(str(demo_path)))

Demo offline: /kaggle/working/evaluasi_041/demo_dan_laporan/demo_evaluasi_041.html
Gambar laporan dan CSV: /kaggle/working/evaluasi_041/demo_dan_laporan


/kaggle/working/evaluasi_041/demo_dan_laporan/demo_evaluasi_041.html

In [66]:
# Eksperimen kecil: pengaruh confidence pada lima foto kegagalan
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from pycocotools import mask as mask_utils

if 'EVAL_DATASETS' not in globals() or 'EVAL_DIR' not in globals():
    raise RuntimeError('Jalankan sel persiapan evaluasi terlebih dahulu.')

exp_root = Path(EVAL_DIR)
exp_gt = EVAL_DATASETS['sendiri']['coco']
exp_pred_path = exp_root / 'yolov8s_seg_640_resmi/sendiri/predictions.json'
exp_cases_path = exp_root / 'demo_dan_laporan/daftar_5_foto.csv'
if not exp_pred_path.exists() or not exp_cases_path.exists():
    raise FileNotFoundError('Hasil evaluasi YOLOv8s 640 dan daftar lima foto harus tersedia.')

exp_predictions = json.loads(exp_pred_path.read_text())
exp_cases = pd.read_csv(exp_cases_path)
exp_rows = []
for exp_case, exp_info in enumerate(exp_cases.to_dict('records'), 1):
    exp_id = int(exp_info['id'])
    exp_anns = [a for a in exp_gt.imgToAnns[exp_id] if not a.get('iscrowd')]
    exp_preds = sorted(
        [p for p in exp_predictions if p['image_id'] == exp_id],
        key=lambda p: p['score'], reverse=True,
    )
    exp_iou = mask_utils.iou(
        [p['segmentation'] for p in exp_preds],
        [exp_gt.annToRLE(a) for a in exp_anns],
        [0] * len(exp_anns),
    ) if exp_preds and exp_anns else np.zeros((len(exp_preds), len(exp_anns)))

    for exp_conf in [0.10, 0.25, 0.50]:
        for exp_threshold in [0.25, 0.50, 0.75]:
            exp_matched = set()
            exp_selected = [i for i, p in enumerate(exp_preds) if p['score'] >= exp_conf]
            for exp_i in exp_selected:
                exp_choices = [
                    j for j, a in enumerate(exp_anns)
                    if j not in exp_matched
                    and a['category_id'] == exp_preds[exp_i]['category_id']
                ]
                if exp_choices:
                    exp_j = max(exp_choices, key=lambda j: exp_iou[exp_i, j])
                    if exp_iou[exp_i, exp_j] >= exp_threshold:
                        exp_matched.add(exp_j)
            exp_tp = len(exp_matched)
            exp_rows.append({
                'case': exp_case, 'file_name': exp_info['file_name'],
                'confidence': exp_conf, 'iou': exp_threshold,
                'GT': len(exp_anns), 'TP': exp_tp,
                'FP': len(exp_selected) - exp_tp, 'FN': len(exp_anns) - exp_tp,
            })

exp_results = pd.DataFrame(exp_rows)
exp_output = exp_root / 'eksperimen_kecil'
exp_output.mkdir(parents=True, exist_ok=True)
exp_results.to_csv(exp_output / 'eksperimen_confidence_dan_iou.csv', index=False)
exp_main = exp_results[exp_results['iou'] == 0.50]
print('Hasil utama: mask IoU 0,50')
display(exp_main[['case', 'confidence', 'GT', 'TP', 'FP', 'FN']])
print('Total pada lima foto')
display(exp_main.groupby('confidence')[['GT', 'TP', 'FP', 'FN']].sum())
print('Diagnosis tambahan: confidence tetap 0,25, ambang IoU diubah')
display(exp_results[exp_results['confidence'] == 0.25][['case', 'iou', 'TP', 'FP', 'FN']])
print('CSV disimpan di:', exp_output)

Hasil utama: mask IoU 0,50


,case,confidence,GT,TP,FP,FN
1,1,0.10,12,1,7,11
4,1,0.25,12,0,2,12
7,1,0.50,12,0,0,12
10,2,0.10,6,0,20,6
13,2,0.25,6,0,9,6
16,2,0.50,6,0,4,6
19,3,0.10,5,0,14,5
22,3,0.25,5,0,5,5
25,3,0.50,5,0,3,5
28,4,0.10,4,1,7,3


Total pada lima foto


,GT,TP,FP,FN
confidence,,,,
0.10,30,3,56,27
0.25,30,0,24,30
0.50,30,0,11,30


Diagnosis tambahan: confidence tetap 0,25, ambang IoU diubah


,case,iou,TP,FP,FN
3,1,0.25,0,2,12
4,1,0.50,0,2,12
5,1,0.75,0,2,12
12,2,0.25,1,8,5
13,2,0.50,0,9,6
14,2,0.75,0,9,6
21,3,0.25,0,5,5
22,3,0.50,0,5,5
23,3,0.75,0,5,5
30,4,0.25,0,3,4


CSV disimpan di: /kaggle/working/evaluasi_041/eksperimen_kecil
Eksperimen memakai prediksi tersimpan; tidak ada training atau inferensi ulang.


## Perbandingan hasil dan domain shift

In [67]:
import shutil

metric_paths = [EVAL_DIR / name / 'metrics.csv' for name, _, _, _ in RUN_CONFIGS]
missing = [str(path) for path in metric_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'Evaluasi belum lengkap: {missing}')
evaluation_table = pd.concat([pd.read_csv(path) for path in metric_paths], ignore_index=True)
evaluation_table.to_csv(EVAL_DIR / 'hasil_evaluasi_semua.csv', index=False)
display(evaluation_table[evaluation_table.class_scope == 'all'])
shared_table = evaluation_table[evaluation_table.class_scope == 'shared']
comparison = shared_table.pivot(index=['model', 'input'], columns='test_set', values='mask_mAP50_95').reset_index()
comparison['delta_sendiri_minus_resmi'] = comparison['sendiri'] - comparison['resmi']
comparison.to_csv(EVAL_DIR / 'domain_shift_shared_classes.csv', index=False)
display(comparison)
# Checkpoint tetap disimpan di Output training. Arsip evaluasi hanya berisi hasil evaluasi.
archive_root = Path('/kaggle/working/arsip_evaluasi_041')
archive_root.mkdir(exist_ok=True)
for item in EVAL_DIR.iterdir():
    if item.name in {'checkpoints', 'test_sendiri', 'dataset_resmi_v5'}:
        continue
    destination = archive_root / item.name
    if item.is_dir():
        shutil.copytree(item, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(item, destination)
archive = shutil.make_archive('/kaggle/working/evaluasi_041_hasil', 'zip', root_dir=archive_root)
print('Selesai: sembilan model/ukuran × dua test set.')
print('ZIP evaluasi:', archive)

,run,model,input,test_set,class_scope,images,mask_mAP50_95,mask_mAP50,mask_AR100,pipeline_ms_per_image
0,yolov8n_seg_640_resmi,YOLOv8n-seg,640,resmi,all,359,0.063393,0.107739,0.180886,118.709440
2,yolov8n_seg_640_resmi,YOLOv8n-seg,640,sendiri,all,44,0.117889,0.163751,0.241635,620.479347
4,yolov8n_seg_320_resmi,YOLOv8n-seg,320,resmi,all,359,0.040024,0.073226,0.126599,49.991094
6,yolov8n_seg_320_resmi,YOLOv8n-seg,320,sendiri,all,44,0.084325,0.142124,0.151326,229.770120
8,yolov8n_seg_960_resmi,YOLOv8n-seg,960,resmi,all,359,0.060576,0.099758,0.183865,135.948518
10,yolov8n_seg_960_resmi,YOLOv8n-seg,960,sendiri,all,44,0.099115,0.140138,0.235043,337.033022
12,yolov8s_seg_640_resmi,YOLOv8s-seg,640,resmi,all,359,0.100488,0.155379,0.239290,88.386527
14,yolov8s_seg_640_resmi,YOLOv8s-seg,640,sendiri,all,44,0.160686,0.231140,0.284923,277.679512
16,yolov8s_seg_320_resmi,YOLOv8s-seg,320,resmi,all,359,0.064965,0.108102,0.174597,50.336197
18,yolov8s_seg_320_resmi,YOLOv8s-seg,320,sendiri,all,44,0.121250,0.189015,0.208676,214.681372


test_set,model,input,resmi,sendiri,delta_sendiri_minus_resmi
0,Mask R-CNN ResNet-50 FPN,320,0.051480,0.117436,0.065956
1,Mask R-CNN ResNet-50 FPN,640,0.080610,0.150407,0.069797
2,Mask R-CNN ResNet-50 FPN,960,0.086610,0.172501,0.085891
3,YOLOv8n-seg,320,0.040173,0.089488,0.049315
4,YOLOv8n-seg,640,0.077811,0.125107,0.047296
5,YOLOv8n-seg,960,0.075700,0.101515,0.025815
6,YOLOv8s-seg,320,0.066543,0.128674,0.062131
7,YOLOv8s-seg,640,0.110590,0.169905,0.059315
8,YOLOv8s-seg,960,0.116663,0.137723,0.021059


Selesai: sembilan model/ukuran × dua test set.
ZIP evaluasi: /kaggle/working/evaluasi_041_hasil.zip
